# Minimal MaxEnt Q-MCMC sampling demo

This notebook shows how to call the MaxEnt MEM Q-based MCMC sampler `sample_mem_distribution_emcee` on a toy MEM-like result dictionary and visualise the resulting confidence band.

In a real workflow you would replace the synthetic result with the output of a MaxEnt solver (e.g. `solve_lifetime_mem` or the higher-level `run_lifetime_mem_from_arrays`).


In [ ]:
import numpy as np

# Import the sampler from the MaxEnt plugin.
#
# If you are running inside a full ChiSurf installation, you can also do:
#   from chisurf.plugins.fluorescence_decay.maxent_decay.fmem import sample_mem_distribution_emcee
# Here we import directly from the fmem.sampling module for clarity.
from chisurf.plugins.fluorescence_decay.maxent_decay.fmem.sampling import sample_mem_distribution_emcee

# --- Build a toy MEM-like result dict -------------------------------------
n = 64
tau = np.linspace(0.1, 5.0, n)  # lifetime axis (ns)

# Start from a simple Gaussian-shaped distribution on the tau grid.
p0 = np.exp(-0.5 * ((tau - 2.5) / 0.3) ** 2)
p0 /= p0.sum()

# For this toy example we use a very simple quadratic form H ≈ I and g0 = H @ p0.
H = np.eye(n)
g0 = H @ p0

result = {
    "tau": tau,
    "p": p0,
    "H": H,
    "g0": g0,
    # Flat prior; in real analyses this is typically 1/tau or a problem-specific prior.
    "prior": np.ones_like(p0),
    # Effective regularisation parameter nu used in Q = chi^2 - 0.5 * nu * S.
    "nu": 1e-3,
}

stats = sample_mem_distribution_emcee(
    result,
    filename=None,          # set to a path like 'mem_samples.h5' to write HDF5 output
    steps_total=200,        # keep this small for a quick demo
    thin=2,
    substeps=50,
    progress_cb=None,       # no GUI progress in this notebook example
)

print("Collected posterior samples:", stats["n_samples"])


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

axis = np.asarray(stats["axis"], dtype=float).ravel()
p_mem = np.asarray(stats["p_mem"], dtype=float).ravel()
p_lo = np.asarray(stats["p_lo"], dtype=float).ravel()
p_hi = np.asarray(stats["p_hi"], dtype=float).ravel()

fig, ax = plt.subplots(figsize=(6, 4))

ax.plot(axis, p_mem, label="MEM distribution p0", color="C0")
ax.fill_between(axis, p_lo, p_hi, color="C1", alpha=0.3, label="Q-MCMC 68% band")

ax.set_xlabel("lifetime (ns)")
ax.set_ylabel("probability")
ax.legend()
ax.set_title("Toy MEM distribution with Q-MCMC confidence band")
plt.tight_layout()
plt.show()
